In [0]:
df = spark.table("bronze_transactions")
bronze_count = df.count()
print("Total bronze records before cleaning:", bronze_count)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, count, when

dup_counts = (
    spark.table("bronze_transactions")
    .groupBy("transaction_id")
    .agg(count("*").alias("cnt"))
    .filter("cnt > 1")
)

dups_removed=dup_counts.count()
print("Total duplicates to be removed:", dups_removed)
print("Total bronze records after removing duplicates:", bronze_count-dups_removed)

window = Window.partitionBy("transaction_id").orderBy(col("ingestion_timestamp").desc())

dedup_df = (
    df.withColumn("row_num", row_number().over(window))
      .filter(col("row_num") == 1)
      .drop("row_num")
)

dedup_flagged = df.withColumn(
    "is_duplicate",
    when(col("row_num") > 1, True).otherwise(False)
)

In [0]:
from pyspark.sql.functions import coalesce, lit

clean_df = (
    dedup_df
    .withColumn("price", coalesce(col("price").cast("double"), lit(0.0)))
    .withColumn("quantity", col("quantity").cast("int"))
)

In [0]:
filtered_df = clean_df.filter(
    (col("price") >= 0) & 
    (col("quantity") > 0)
)

In [0]:
from pyspark.sql.functions import when

final_df = filtered_df.withColumn(
    "data_quality_flag",
    when(col("price") == 0, "price_imputed")
    .otherwise("clean")
)

In [0]:
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_transactions")

rejected_df = clean_df.filter(
    (col("price") < 0) | (col("quantity") <= 0)
)
rejected_df.write.format("delta").mode("overwrite").saveAsTable("rejected_transactions")

In [0]:
silver_count=spark.sql("SELECT COUNT(*) FROM silver_transactions").collect()[0][0]

recs_removed=bronze_count-silver_count

print("Total bronze records before cleaning:", bronze_count)
print("Total silver records after cleaning:", silver_count)
print("Total records removed:", recs_removed)
print("Percentage of records removed:", round(recs_removed/bronze_count*100,2),"%")
print("Duplicate records removed:", dups_removed)
print("Records in rejected_transactions:", spark.sql("SELECT COUNT(*) FROM rejected_transactions").collect()[0][0])

display(spark.table("silver_transactions"))